In [15]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pprint import pprint

## Fetching & Exploring an Bundestag plenary Seesion xml file
To start exploring we will try to fetch an xml file, and navigate it as a tree.
Once we are comfortable navigating the file we'll try flattening it into a flat table
/ a flat pandas df as the first step of denormilization (flattening a nested file structure),
so that we can after further normalize into different tables.

In [16]:
file_path ="https://www.bundestag.de/resource/blob/1140642/21057.xml"

response = requests.get(file_path)
response.raise_for_status()
print(response.status_code)
print(type(response))


200
<class 'requests.models.Response'>


In [17]:
# after fetching the xml using an https request
# we can access it as an tree by  converting it to
# an elementtree object, which is easier to navigate/work with
# for that we use pythons built in xml ElementTree object
# which represents an xml file, as the name suggests, as a tree
# of elements. We imported this type of object already so we can
# can make use of that class to instantiate an instance that holds
# the information of our plenary protocoll.

# For that we use the fromstring method, which expects, as the name suggests, as string
# as input. To access the our html response to a string, we can access its content attribute.

# Each tree has one core node, also referred to as the trunk of a tree.
# In this case we refer to it as the root of our element tree.
# We now pass our xml as a string, and get an elementree that we assign
# to root
# we see that we succesfully get an Elementree element (as the root is always returned, which holds
# all subelements/children of the rest of the tree), remember we're working with a hirarchical # datastructure here, we can access the root elements name by accessing its tag attribute,
# as well as see that the root holds 4 subelements which are its children
root = ET.fromstring(response.content)
print(type(root))
print(root.tag)
print(len(root))

<class 'xml.etree.ElementTree.Element'>
dbtplenarprotokoll
4


In [18]:
# in addition to that an element is of a certain type (its name), can hold on to values, usually text
# as well as can have attributes
# In our case our root element is called "dbtplenarprotokoll" which translates to "dbtplenaryprotocol"
# but holds varius attributes as metadata
# those attributes are returned as a python dictionary, which means that we can access
# them using their keys
# We will make use of that to get our plenary level metadata like session date, legislative
# period, session number etc
print(root.attrib)
print(type(root.attrib))
plenary_session_metadata = root.attrib

{'vertrieb': 'Bundesanzeiger Verlag GmbH, Postfach 1 0 05 34, 50445 Köln, Telefon (02 21) 97 66 83 40, Fax (02 21) 97 66 83 44, www.bundesanzeiger-verlag.de', 'herstellung': 'H. Heenemann GmbH  Co. KG, Buch- und Offsetdruckerei, Bessemerstraße 83–91, 12103 Berlin, www.heenemann-druck.de', 'sitzung-ort': 'Berlin', 'herausgeber': 'Deutscher Bundestag', 'issn': '0722-7980', 'wahlperiode': '21', 'sitzung-nr': '57', 'sitzung-datum': '30.01.2026', 'sitzung-start-uhrzeit': '09:00', 'sitzung-ende-uhrzeit': '15:00', 'sitzung-naechste-datum': '25.02.2026', 'start-seitennr': '6839'}
<class 'dict'>


In [19]:
# to test we try to access the legaslative period, session nr and
# date, later when flattening the file we can make use of this to
# add session level metadata to each speech
legaslative_period = plenary_session_metadata["wahlperiode"]
session_nr = plenary_session_metadata["sitzung-nr"]
session_date = plenary_session_metadata["sitzung-datum"]
print(f"Legaslative Period: {legaslative_period}")
print(f"Session Nr: {session_nr}")
print(f"Session Date: {session_date}")

Legaslative Period: 21
Session Nr: 57
Session Date: 30.01.2026


In [20]:
# ok now that we've covered everyhting of the root node
# we can move down the tree and explore its sub elements, also
# referred to as its child
for child in root:
    print(child.tag)

# in our case the root holds 4 children:
# the vorspann, the actual undergoings of the session in
# sitzungsverlauf which also hold the individual speeches,
# additional information in anlagen
# and lastly a speakerlist, which later might be useful to validate
# our extracted speeches against
# Of most interest to us is the sitzungsverlauf which contains
# the core of our speeches


vorspann
sitzungsverlauf
anlagen
rednerliste


In [33]:
# Among other ways the two most straightforward ways of accessing sub-elements/children of an xml element
# are accessing them by their location (index) or by their tag.
# The index is used like with any other iterable (like a list), to get elements
# of a certain tag we can use the find or findall methods
print(f"Index: {root[0].tag}\nTag: {root.find("vorspann").tag}")


Index: vorspann
Tag: vorspann


In [34]:
# for convenience we will get hold of the 4 main sections
# of the plenary protocol
opening = root.find("vorspann")
agenda = root.find("sitzungsverlauf")
attachment = root.find("anlagen")
speaker_list = root.find("rednerliste")

# lets check
print(f"Opening: {opening.tag}")
print(f"Agenda: {agenda.tag}")
print(f"Attachment: {attachment.tag}")
print(f"Opening: {speaker_list.tag}")


Opening: vorspann
Agenda: sitzungsverlauf
Attachment: anlagen
Opening: rednerliste


In [36]:
# As the core of the session of our interest is the main part
# as it holds all our speeches, lets dig deeper and check which subelements
# it consists of
print(f"Nr. of main elements: {len(agenda)}\n")

print("Main elements (Name and attributes): \n")
for child in agenda:
     print(child.tag)
     print(child.attrib)

# What wenotice is that the agenda containts and opening and closing,
# as well as agenda items which represent debates

Nr. of main elements: 9

Main elements (Name and attributes): 

sitzungsbeginn
{'sitzung-start-uhrzeit': '09:00'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 7'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 24'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt'}
tagesordnungspunkt
{'top-id': 'Zusatzpunkt 8'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 5'}
tagesordnungspunkt
{'top-id': 'Tagesordnungspunkt 26'}
tagesordnungspunkt
{'top-id': 'Zusatzpunkt 9'}
sitzungsende
{'sitzung-ende-uhrzeit': '15:00'}


In [39]:
# lets get hold of the agenda items as a list
# and explore the tags, attributes and nr. of subelements of the
# first agenda item
agenda_items = agenda.findall("tagesordnungspunkt")
first_agenda_item = agenda_items[0]
print(f"Agenda item name: {first_agenda_item.tag}")
print(f"Agenda attributes: {first_agenda_item.attrib}")
print(f"Nr. of sub elements: {len(first_agenda_item)}")

# We find the tag name, as expected, the agenda items title as the top-id attribute
# and 33 sub-elements, which the majority of will be the speeches of that debate

Agenda item name: tagesordnungspunkt
Agenda attributes: {'top-id': 'Tagesordnungspunkt 7'}
Nr. of sub elements: 33


In [47]:
# lets check what sub elements
# the agenda item holds
for item in first_agenda_item:
    print(item.tag)

# We find a lot of p tags (paragraphs), many "rede" tags (translates to speech)
# and few "kommentar" tags (translates to comment). Manual investigations showed that first
# paragraphs often  serve as a debate opening that entails the title, sub elements fo the debate
# and sometimes documents that the debate refers to.
# This is followed by speeches and closing paragraphs. The comment tags seem to be the exception rather
# than the norm, but need to be further investigated

p
p
p
p
p
p
p
p
p
kommentar
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
rede
name
p
p
kommentar
rede
rede
name
p
p
p


In [49]:
# Lets keep our focus with the first agenda item and dig deeper
# For this lets get all paragraphs and all speeches
paragraphs = first_agenda_item.findall("p")
speeches = first_agenda_item.findall("rede")
nr_speeches = len(speeches)
print(f"Nr. of paragraphs: {len(paragraphs)}")
print(f"Nr. of speeches: {nr_speeches}")

Nr. of paragraphs: 14
Nr. of speeches: 15


In [54]:
# It turns out that most of the paragraphs are actually leaves of our element tree,
# meaning that they have no sub-elements/children and thus present the last nodes of that particular branch.
# Lets explore them and their attributes which also surfaces that we need another method
# to target specific paragraphs and lastly get their values
for p in paragraphs:
    print(f"Paragraph Attributes: {p.attrib}")
    print(f"Paragraph values: {p.text}")

# We see that all pargraphs hold a "klasse" attributes (translates to class)
# All p's with class values of "T_fett" hold the title of the agenda item
# The ones with "T_NaS" seem to hold subtitles
# If the agenda relates to documents, we find a p of class "T_Drs"
#  that hold an sub anchor elements with an urlpath that links to that document

Paragraph Attributes: {'klasse': 'J'}
Paragraph values: Ich rufe die Tagesordnungspunkte 7a und 7b auf: 
Paragraph Attributes: {'klasse': 'T_NaS'}
Paragraph values: 	7	a)	Abgabe einer Regierungserklärung durch die Bundesministerin für Wirtschaft und Energie: 
Paragraph Attributes: {'klasse': 'T_fett'}
Paragraph values: 			Zum Jahreswirtschaftsbericht 2026
Paragraph Attributes: {'klasse': 'T_NaS'}
Paragraph values: 		b)	Beratung der Unterrichtung durch die Bundesregierung 
Paragraph Attributes: {'klasse': 'T_fett'}
Paragraph values: 			Jahreswirtschaftsbericht 2026 der Bundesregierung
Paragraph Attributes: {'klasse': 'T_Drs'}
Paragraph values: 			Drucksache 
Paragraph Attributes: {'klasse': 'T_Ueberweisung'}
Paragraph values: 			Überweisungsvorschlag: Ausschuss für Wirtschaft und Energie (f) Ausschuss für Recht und Verbraucherschutz Finanzausschuss Ausschuss für Landwirtschaft, Ernährung und Heimat Ausschuss für Arbeit und Soziales Ausschuss für Bildung, Familie, Senioren, Frauen und Ju

In [62]:
# Lets get hold on the titles and document url
main_title = first_agenda_item.findall("p[@klasse='T_fett']")
for title in main_title:
    print(title.text)

			Zum Jahreswirtschaftsbericht 2026
			Jahreswirtschaftsbericht 2026 der Bundesregierung


In [126]:
# lets get hold of sub-elements
# ok lets explore a new method/concept that helps us to
# identify specific elements of our interest, by not only
# defining the tag to look for but also the attribute and if
# needed also the hierarchy level. We can still use find or findall
# but this time we not only the tagname but an xpath.
# When looking for a a p element in the children of the
# first agenda item with the attribute "klasse": "T_NaS"
sub_titles = first_agenda_item.findall("p[@klasse = 'T_fett']")
for title in sub_titles:
    print(title.text)
# In our case we looked for the title(s) of the first agenda item
# which is about the yearly economic report of 2026

			Zum Jahreswirtschaftsbericht 2026
			Jahreswirtschaftsbericht 2026 der Bundesregierung


In [65]:
# Important to know: findall returns an empty list
# if no element matches the defined mask/filter
ok = first_agenda_item.findall("p[@klasse= 'lennart']")
print(ok)

[]


In [127]:
# We can also further define multiple hierarchy levels to navigate
# through by adding backslashes how we would with a filepath too
# We make use of that to look for the document element that
# the first agenda item refers to.
# Further we find a new method that lets us access attributes of an
# element using their attribute name as a key, the get method.
# It lets us specify which attribute to look for and which value
# to return when that attribute is not found
document = first_agenda_item.findall("p[@klasse = 'T_Drs']/a")
print(document_url.get("href", None))
# This time we get hold of the documents url that the first agenda item relates
# too. We saw earlier that the first agenda item is dedicated towards the
# yearly economic report of 2026, the link we just accessed lead to that
# exact report.

https://dserver.bundestag.de/btd/21/037/2103700.pdf


In [128]:
# For testing purposes lets loop over all agenda items
# and try to get the respective doc urls they relate to.
for item in agenda_items:
    documents = item.findall("p[@klasse = 'T_Drs']/a")
    for doc in documents:
        print(doc.get("href", "No URL"))

https://dserver.bundestag.de/btd/21/037/2103700.pdf
https://dserver.bundestag.de/btd/21/036/2103661.pdf
https://dserver.bundestag.de/btd/21/038/2103842.pdf
https://dserver.bundestag.de/btd/21/038/2103843.pdf
https://dserver.bundestag.de/btd/21/036/2103619.pdf
https://dserver.bundestag.de/btd/21/028/2102804.pdf


In [135]:
# Ok the wrap our paragraph exploration, lets get the title, sub title(s) and document links,
# test for all agenda items, so that we can move on to exploring speeches.
# NOTE: We will for now ignore agenda item openings made by the president, which usually is a short introduction
#       of what is to be debated etc.

titles = first_agenda_item.findall("p[@klasse= 'T_fett']")
sub_titles = first_agenda_item.findall(("p[@klasse = 'T_NaS']"))
documents = first_agenda_item.findall("p[@klasse='T_Drs']/a")

print("Title(s):")
for title in titles:
    print(f"Agenda Item Titel: {title.text}")

print("\nSubtitle(s):")
for subtitle in sub_titles:
    print(f"Agenda Item Titel: {subtitle.text}")

print("\nDocument link(s):")

for doc in documents:
    print(f"Document link: {doc.get('href', 'No link found')}")

Title(s):
Agenda Item Titel: 			Zum Jahreswirtschaftsbericht 2026
Agenda Item Titel: 			Jahreswirtschaftsbericht 2026 der Bundesregierung

Subtitle(s):
Agenda Item Titel: 	7	a)	Abgabe einer Regierungserklärung durch die Bundesministerin für Wirtschaft und Energie: 
Agenda Item Titel: 		b)	Beratung der Unterrichtung durch die Bundesregierung 

Document link(s):
Document link: https://dserver.bundestag.de/btd/21/037/2103700.pdf


In [138]:
# For testing purposes let extend this to all agenda items
for item in agenda_items:
    print(50*"-")
    print(f"Agenda item: {item.tag}")

    titles = item.findall("p[@klasse= 'T_fett']")
    sub_titles = item.findall(("p[@klasse = 'T_NaS']"))
    documents = item.findall("p[@klasse='T_Drs']/a")

    print("\nTitle(s):")
    for title in titles:
        print(f"Agenda Item Titel: {title.text}")

    print("\nSubtitle(s):")
    for subtitle in sub_titles:
        print(f"Agenda Item Titel: {subtitle.text}")

    print("\nDocument link(s):")

    for doc in documents:
        print(f"Document link: {doc.get('href', 'No link found')}")

--------------------------------------------------
Agenda item: tagesordnungspunkt

Title(s):
Agenda Item Titel: 			Zum Jahreswirtschaftsbericht 2026
Agenda Item Titel: 			Jahreswirtschaftsbericht 2026 der Bundesregierung

Subtitle(s):
Agenda Item Titel: 	7	a)	Abgabe einer Regierungserklärung durch die Bundesministerin für Wirtschaft und Energie: 
Agenda Item Titel: 		b)	Beratung der Unterrichtung durch die Bundesregierung 

Document link(s):
Document link: https://dserver.bundestag.de/btd/21/037/2103700.pdf
--------------------------------------------------
Agenda item: tagesordnungspunkt

Title(s):
Agenda Item Titel: 		Mobilitätsgarantie einführen – Produktionskapazitäten für die Verkehrswende aufbauen
Agenda Item Titel: 		Gemeindeverkehrsfinanzierungsgesetz novellieren – Kommunen stärken und Ausbau des öffentlichen Personennahverkehrs langfristig absichern

Subtitle(s):
Agenda Item Titel: 	24	Beratung des Antrags der Abgeordneten Luigi Pantisano, Marcel Bauer, Lorenz Gösta Beutin, w

In [132]:
speech = speeches[9]

speaker_info = speech.find("p[@klasse='redner']")

In [112]:
print(len(speaker_info))

2


In [120]:
name = speaker_info.find(".//vorname")
lastname = speaker_info.find(".//nachname")
role = speaker_info.find(".//rolle_lang")

In [122]:
print(name.text)
print(lastname.text)
if role:
    print(role.text)

Sandra
Detzer
